# 저장된 DL checkpoint 평가

`evaluation/eval.yaml`에서 `evaluation.target: dl`, `model_path: ../DL/models/<run_id>/best_model.pt`, `artifact_dir: null`, 평가 split과 MLflow 설정을 지정합니다. 같은 run 폴더의 `metadata.json`, `resolved_config.yaml`이 필요합니다.

저장된 가중치·구조·전처리·threshold를 불러와 환자 단위 확률을 평가합니다. 이 노트북은 모델 학습이나 validation threshold 재선택을 하지 않습니다. Raw run은 기존 sample-rate/quality 인덱스 정합성 조건을 검사합니다.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().resolve()
while not (repo / "Members/sang/model/evaluation/eval.yaml").is_file():
    if repo.parent == repo:
        raise RuntimeError("저장소 내부에서 노트북을 실행하세요")
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
eval_config = repo / "Members/sang/model/evaluation/eval.yaml"

In [ ]:
from Members.sang.model.config import load_yaml
from Members.sang.model.evaluation.evaluation import evaluate_saved_model

if load_yaml(eval_config)["evaluation"]["target"] != "dl":
    raise ValueError("eval.yaml의 evaluation.target을 dl로 설정하세요")
evaluation_run = evaluate_saved_model(eval_config)
print(evaluation_run)
print(json.loads((evaluation_run / "metrics.json").read_text()))
display(pd.read_csv(evaluation_run / "predictions.csv").head())
if (evaluation_run / "mlflow_status.json").is_file():
    print(json.loads((evaluation_run / "mlflow_status.json").read_text()))